# Solvability classifier

Train on `train-3000` + `abstention-pack`. Report metrics on that mix and on `test-100`. `test-100` is not used to fit the model or to overwrite `tau.json`.


In [7]:
from pathlib import Path
import importlib.util
import json

SKLEARN_PIN = "scikit-learn==1.9.1"

# Kaggle mounts this dataset as
# /kaggle/input/datasets/<user>/<slug>/<pack>/<pack>/<task>/task.json
# Local runs still use task_packs/.
SLUGS = {
    "train": ("train-3000", "train_3000"),
    "abstention": ("abstention-pack", "abstention_pack"),
    "test": ("test-100", "test_100"),
}


def _repo_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "harness" / "features.py").is_file() and (cand / "task_packs").is_dir():
            return cand
    return None


def _task_jsons(root: Path) -> list[Path]:
    direct = sorted(p for p in root.glob("*/task.json") if p.is_file())
    if direct:
        return direct
    return sorted(p for p in root.glob("*/*/task.json") if p.is_file())


def _matches_slug(path: Path, slugs: tuple[str, ...]) -> bool:
    blob = path.as_posix().lower()
    return any(slug in blob for slug in slugs)


def find_pack(key: str) -> Path:
    slugs = SLUGS[key]
    roots = []
    kaggle = Path("/kaggle/input")
    if kaggle.is_dir():
        roots.append(kaggle)
    repo = _repo_root()
    if repo is not None:
        roots.append(repo / "task_packs")
    best = None
    best_n = -1
    for root in roots:
        if not root.is_dir():
            continue
        for manifest in root.rglob("manifest.jsonl"):
            parent = manifest.parent
            if not _matches_slug(parent, slugs):
                continue
            n = len(_task_jsons(parent))
            if n > best_n:
                best = parent
                best_n = n
    if best is None or best_n <= 0:
        raise FileNotFoundError(
            f"{key} pack not found under /kaggle/input or task_packs. "
            f"Expected a manifest.jsonl whose path contains one of {slugs}."
        )
    return best


_EMBEDDED_FEATURES = "\"\"\"Tabular features and structural skip flags. No LLM, no expected/reward.\n\nStandalone: notebooks load this file by path, so it must not import the harness\npackage (that pulls openJiuwen).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport ast\nimport re\nfrom pathlib import Path\nfrom typing import Any, Optional\n\nFEATURE_SCHEMA_VERSION = 1\n\n# Bands match harness.estimate. The heuristic stays here so Kaggle and the\n# runtime share one formula when no cost model is loaded.\nCOST_SHORT = 800\nCOST_CODING = 2500\n\nFEATURE_NAMES: tuple[str, ...] = (\n    \"is_math\",\n    \"is_data_analysis\",\n    \"is_search\",\n    \"is_coding\",\n    \"instr_chars\",\n    \"instr_words\",\n    \"instr_lines\",\n    \"digit_ratio\",\n    \"n_numbers\",\n    \"n_questions\",\n    \"has_latex\",\n    \"has_code_fence\",\n    \"mentions_answer_txt\",\n    \"da_named_file_missing\",\n    \"da_no_csv\",\n    \"da_broken_header\",\n    \"da_column_missing\",\n    \"search_kb_empty\",\n    \"search_kb_tiny\",\n    \"search_token_overlap\",\n    \"coding_no_mylib\",\n    \"coding_empty_test\",\n    \"coding_foreign_import\",\n    \"coding_ast_fail\",\n    \"math_arith_steps\",\n    \"csv_lines\",\n    \"csv_cols\",\n    \"kb_files\",\n    \"kb_chars\",\n    \"code_lines\",\n    \"code_defs\",\n    \"code_asserts\",\n)\n\n# Hard skip only when a missing or wrong file makes the task impossible.\n# A syntax error in mylib.py is a feature, not a skip: it may be the bug to fix.\n# Tiny kb is a feature, not a skip: a short real note would be a false skip.\n_HARD_SKIP_FLAGS: dict[str, tuple[str, ...]] = {\n    \"data_analysis\": (\n        \"da_named_file_missing\",\n        \"da_no_csv\",\n    ),\n    \"search\": (\"search_kb_empty\",),\n    \"coding\": (\n        \"coding_no_mylib\",\n        \"coding_empty_test\",\n        \"coding_foreign_import\",\n    ),\n}\n\n_FILE_RE = re.compile(r\"\\b([\\w./-]+\\.(?:csv|tsv|json|txt|md|py))\\b\", re.IGNORECASE)\n_COL_RE = re.compile(r\"\\bcolumn\\s+([A-Za-z_][\\w]*)\", re.IGNORECASE)\n_NUM_RE = re.compile(r\"\\d+(?:\\.\\d+)?\")\n_WORD_RE = re.compile(r\"[A-Za-z]{3,}\")\n_IMPORT_RE = re.compile(\n    r\"^\\s*(?:from|import)\\s+([A-Za-z_][\\w]*)\",\n    re.MULTILINE,\n)\n_ARITH_RE = re.compile(\n    r\"\\b(?:split|double|doubled|add|added|receive|receives|ship|ships|\"\n    r\"merge|merged|minus|plus|times|divided|subtract|multiply)\\b\",\n    re.IGNORECASE,\n)\n_DEF_RE = re.compile(r\"^\\s*def\\s+\\w+\", re.MULTILINE)\n_ASSERT_RE = re.compile(r\"\\bassert\\b|\\bdef\\s+test_\\w+\", re.IGNORECASE)\n\n_STOP = {\n    \"the\", \"and\", \"for\", \"with\", \"that\", \"this\", \"from\", \"are\", \"was\", \"were\",\n    \"you\", \"your\", \"only\", \"file\", \"answer\", \"when\", \"done\", \"write\", \"final\",\n    \"current\", \"directory\", \"example\", \"bash\", \"echo\", \"python\", \"compute\",\n    \"work\", \"inside\", \"using\", \"use\", \"what\", \"which\", \"who\", \"how\", \"many\",\n    \"does\", \"have\", \"has\", \"into\", \"then\", \"than\", \"after\", \"before\", \"about\",\n}\n\n_CSV_LINE_CAP = 5000\n_KB_CHAR_CAP = 400_000\n_KB_TINY_CHARS = 200\n\n\ndef _task_type(task: Any) -> str:\n    raw = getattr(task, \"task_type\", \"\")\n    return raw.value if hasattr(raw, \"value\") else str(raw)\n\n\ndef _work_dir(task: Any) -> Path:\n    work = getattr(task, \"work_dir\", None)\n    if work is not None:\n        return Path(work)\n    return Path(getattr(task, \"task_dir\"))\n\n\ndef _read_text(path: Path, cap: int) -> str:\n    try:\n        data = path.read_text(encoding=\"utf-8\", errors=\"replace\")\n    except OSError:\n        return \"\"\n    return data[:cap]\n\n\ndef _named_files(instruction: str) -> list[str]:\n    found: list[str] = []\n    for match in _FILE_RE.findall(instruction):\n        if Path(match).name.lower() == \"answer.txt\":\n            continue\n        found.append(match)\n    return found\n\n\ndef _file_exists(work: Path, name: str) -> bool:\n    if (work / name).is_file():\n        return True\n    return (work / Path(name).name).is_file()\n\n\ndef _table_paths(work: Path) -> list[Path]:\n    paths: list[Path] = []\n    for pattern in (\"*.csv\", \"*.tsv\", \"*.json\"):\n        paths.extend(p for p in work.glob(pattern) if p.is_file())\n    return sorted(paths)\n\n\ndef _csv_shape(path: Path) -> tuple[int, int, bool, list[str]]:\n    \"\"\"Return line count (capped), column count, broken-header flag, header cells.\"\"\"\n    try:\n        fh = path.open(\"r\", encoding=\"utf-8\", errors=\"replace\")\n    except OSError:\n        return 0, 0, True, []\n    lines = 0\n    header: list[str] = []\n    with fh:\n        for line in fh:\n            if lines == 0:\n                header = [cell.strip() for cell in line.rstrip(\"\\n\").split(\",\")]\n            lines += 1\n            if lines >= _CSV_LINE_CAP:\n                break\n    broken = lines == 0 or not any(header)\n    cols = 0 if broken else len(header)\n    return lines, cols, broken, header\n\n\ndef _kb_stats(work: Path) -> tuple[int, int, str]:\n    kb = work / \"kb\"\n    if not kb.is_dir():\n        return 0, 0, \"\"\n    files = sorted(p for p in kb.glob(\"*.md\") if p.is_file())\n    chunks: list[str] = []\n    total = 0\n    for path in files:\n        text = _read_text(path, _KB_CHAR_CAP - total)\n        if not text:\n            continue\n        chunks.append(text)\n        total += len(text)\n        if total >= _KB_CHAR_CAP:\n            break\n    body = \"\\n\".join(chunks)\n    nonempty = sum(1 for chunk in chunks if chunk.strip())\n    return nonempty, len(body), body\n\n\ndef _token_overlap(instruction: str, kb_text: str) -> float:\n    question = {w.lower() for w in _WORD_RE.findall(instruction) if w.lower() not in _STOP}\n    if not question or not kb_text:\n        return 0.0\n    kb_words = {w.lower() for w in _WORD_RE.findall(kb_text)}\n    return len(question & kb_words) / len(question)\n\n\ndef _coding_stats(work: Path) -> dict[str, float]:\n    lib = work / \"mylib.py\"\n    test = work / \"test_mylib.py\"\n    lib_text = _read_text(lib, 200_000) if lib.is_file() else \"\"\n    test_text = _read_text(test, 200_000) if test.is_file() else \"\"\n    ast_fail = 0.0\n    if lib.is_file():\n        try:\n            ast.parse(lib_text)\n        except SyntaxError:\n            ast_fail = 1.0\n    imports = {name for name in _IMPORT_RE.findall(test_text)}\n    foreign = 1.0 if imports and \"mylib\" not in imports else 0.0\n    return {\n        \"coding_no_mylib\": 0.0 if lib.is_file() else 1.0,\n        \"coding_empty_test\": 0.0 if test_text.strip() else 1.0,\n        \"coding_foreign_import\": foreign,\n        \"coding_ast_fail\": ast_fail,\n        \"code_lines\": float(lib_text.count(\"\\n\") + (1 if lib_text.strip() else 0)),\n        \"code_defs\": float(len(_DEF_RE.findall(lib_text))),\n        \"code_asserts\": float(len(_ASSERT_RE.findall(test_text))),\n    }\n\n\ndef extract(task: Any) -> dict[str, float]:\n    \"\"\"Feature dict keyed by FEATURE_NAMES. Reward is intentionally absent.\"\"\"\n    task_type = _task_type(task)\n    instruction = str(getattr(task, \"instruction\", \"\") or \"\")\n    work = _work_dir(task)\n    chars = len(instruction)\n    words = re.findall(r\"\\S+\", instruction)\n\n    feats = {name: 0.0 for name in FEATURE_NAMES}\n    feats[\"is_math\"] = 1.0 if task_type == \"math\" else 0.0\n    feats[\"is_data_analysis\"] = 1.0 if task_type == \"data_analysis\" else 0.0\n    feats[\"is_search\"] = 1.0 if task_type == \"search\" else 0.0\n    feats[\"is_coding\"] = 1.0 if task_type == \"coding\" else 0.0\n    feats[\"instr_chars\"] = float(chars)\n    feats[\"instr_words\"] = float(len(words))\n    feats[\"instr_lines\"] = float(instruction.count(\"\\n\") + (1 if instruction else 0))\n    feats[\"digit_ratio\"] = (sum(ch.isdigit() for ch in instruction) / chars) if chars else 0.0\n    feats[\"n_numbers\"] = float(len(_NUM_RE.findall(instruction)))\n    feats[\"n_questions\"] = float(instruction.count(\"?\"))\n    feats[\"has_latex\"] = 1.0 if (\"$\" in instruction or \"\\\\frac\" in instruction) else 0.0\n    feats[\"has_code_fence\"] = 1.0 if \"```\" in instruction else 0.0\n    feats[\"mentions_answer_txt\"] = 1.0 if \"answer.txt\" in instruction.lower() else 0.0\n    feats[\"math_arith_steps\"] = float(len(_ARITH_RE.findall(instruction)))\n\n    if task_type == \"data_analysis\":\n        named = _named_files(instruction)\n        if named and any(not _file_exists(work, name) for name in named):\n            feats[\"da_named_file_missing\"] = 1.0\n        tables = _table_paths(work)\n        if not tables:\n            feats[\"da_no_csv\"] = 1.0\n        else:\n            preferred = tables[0]\n            for name in named:\n                candidate = work / Path(name).name\n                if candidate.is_file():\n                    preferred = candidate\n                    break\n            if preferred.suffix.lower() in {\".csv\", \".tsv\"}:\n                lines, cols, broken, header = _csv_shape(preferred)\n                feats[\"csv_lines\"] = float(lines)\n                feats[\"csv_cols\"] = float(cols)\n                feats[\"da_broken_header\"] = 1.0 if broken else 0.0\n                header_key = {cell.casefold() for cell in header if cell}\n                missing_col = [\n                    col for col in _COL_RE.findall(instruction)\n                    if col.casefold() not in header_key\n                ]\n                if missing_col:\n                    feats[\"da_column_missing\"] = 1.0\n\n    if task_type == \"search\":\n        n_files, n_chars, body = _kb_stats(work)\n        feats[\"kb_files\"] = float(n_files)\n        feats[\"kb_chars\"] = float(n_chars)\n        feats[\"search_kb_empty\"] = 1.0 if n_files == 0 or not body.strip() else 0.0\n        feats[\"search_kb_tiny\"] = 1.0 if 0 < n_chars < _KB_TINY_CHARS else 0.0\n        feats[\"search_token_overlap\"] = _token_overlap(instruction, body)\n\n    if task_type == \"coding\":\n        feats.update(_coding_stats(work))\n\n    return feats\n\n\ndef vector(task: Any) -> list[float]:\n    feats = extract(task)\n    return [float(feats[name]) for name in FEATURE_NAMES]\n\n\ndef structural_skip_reason(task: Any, feats: Optional[dict[str, float]] = None) -> Optional[str]:\n    \"\"\"Comma-joined hard-fail flags, or None. Math never hard-fails.\"\"\"\n    task_type = _task_type(task)\n    flags = _HARD_SKIP_FLAGS.get(task_type)\n    if not flags:\n        return None\n    row = feats if feats is not None else extract(task)\n    hit = [name for name in flags if row.get(name, 0.0) >= 1.0]\n    if not hit:\n        return None\n    return \",\".join(hit)\n\n\ndef heuristic_cost(task: Any, feats: Optional[dict[str, float]] = None) -> int:\n    \"\"\"Same bands as harness.estimate.feature_adjust, from the feature row.\"\"\"\n    row = feats if feats is not None else extract(task)\n    base = COST_CODING if row[\"is_coding\"] >= 1.0 else COST_SHORT\n    factor = 1.0\n    if row[\"instr_chars\"] > 800:\n        factor *= 1.2\n    if row[\"is_search\"] >= 1.0 and row[\"kb_files\"] > 100:\n        factor *= 1.1\n    if row[\"is_data_analysis\"] >= 1.0 and row[\"csv_lines\"] > 500:\n        factor *= 1.1\n    if row[\"is_coding\"] >= 1.0 and row[\"coding_no_mylib\"] < 1.0:\n        factor *= 1.0 + min(row[\"code_lines\"], 200.0) / 200.0\n    return max(1, int(base * factor))\n"

def find_features() -> Path:
    candidates = []
    repo = _repo_root()
    if repo is not None:
        candidates.append(repo / "harness" / "features.py")
    kaggle = Path("/kaggle/input")
    if kaggle.is_dir():
        candidates.extend(kaggle.rglob("features.py"))
    for path in candidates:
        if not path.is_file():
            continue
        head = path.read_text(encoding="utf-8", errors="replace")[:800]
        if "FEATURE_SCHEMA_VERSION" in head:
            return path
    dest = Path("/kaggle/working/features.py")
    if not dest.parent.is_dir():
        dest = Path("features.py")
    dest.write_text(_EMBEDDED_FEATURES, encoding="utf-8")
    print("features.py is not in the Kaggle dataset; using the copy stored in this notebook")
    return dest


PACKS = {key: find_pack(key) for key in SLUGS}
FEATURES_PY = find_features()

spec = importlib.util.spec_from_file_location("rank_features", FEATURES_PY)
features = importlib.util.module_from_spec(spec)
spec.loader.exec_module(features)
print("features", FEATURES_PY, "schema", features.FEATURE_SCHEMA_VERSION)
print({k: str(v) for k, v in PACKS.items()})


def out_dir() -> Path:
    kaggle_out = Path("/kaggle/working")
    if kaggle_out.is_dir():
        return kaggle_out
    return Path.cwd()


OUT = out_dir()
print("artifact dir", OUT)


features.py is not in the Kaggle dataset; using the copy stored in this notebook
features /kaggle/working/features.py schema 1
{'train': '/kaggle/input/datasets/danbrag/agent-dataset/train-3000/train-3000', 'abstention': '/kaggle/input/datasets/danbrag/agent-dataset/abstention-pack/abstention-pack', 'test': '/kaggle/input/datasets/danbrag/agent-dataset/test-100/test-100'}
artifact dir /kaggle/working


In [8]:
def load_pack(root: Path) -> list[dict]:
    manifest = {}
    with (root / "manifest.jsonl").open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            manifest[row["task_id"]] = row
    rows = []
    for task_json in _task_jsons(root):
        raw = json.loads(task_json.read_text(encoding="utf-8"))
        task_dir = task_json.parent
        work = task_dir / "workspace"

        class _Task:
            pass

        task = _Task()
        task.task_id = str(raw["task_id"])
        task.task_type = str(raw["task_type"])
        task.instruction = str(raw["instruction"])
        task.reward = int(raw["reward"])
        task.work_dir = work if work.is_dir() else task_dir
        feats = features.extract(task)
        meta = manifest.get(task.task_id, {})
        hard = features.structural_skip_reason(task, feats)
        rows.append({
            **feats,
            "task_id": task.task_id,
            "task_type": task.task_type,
            "reward": task.reward,
            "solvable": bool(meta.get("solvable", False)),
            "hard_skip": hard is not None,
            "hard_reason": hard or "",
            "cost_hat": features.heuristic_cost(task, feats),
        })
    return rows


train_rows = load_pack(PACKS["train"]) + load_pack(PACKS["abstention"])
test_rows = load_pack(PACKS["test"])
print(f"train-mix {len(train_rows)} solvable {sum(r['solvable'] for r in train_rows)}")
print(f"test-100 {len(test_rows)} solvable {sum(r['solvable'] for r in test_rows)}")


<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\W'
<unknown>:3: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\*'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\d'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:3: SyntaxWarning: invalid escape sequence '\.'
<unknown>:2: SyntaxWarning: invalid escape sequence '\.'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'


train-mix 3100 solvable 2400
test-100 100 solvable 80


In [9]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

NAMES = list(features.FEATURE_NAMES)
train_df = pd.DataFrame(train_rows)
test_df = pd.DataFrame(test_rows)
X_train = train_df[NAMES]
y_train = train_df["solvable"].astype(int)

clf = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.08,
    max_iter=200,
    random_state=0,
)
clf.fit(X_train, y_train)
train_df["p"] = clf.predict_proba(X_train)[:, list(clf.classes_).index(1)]
test_df["p"] = clf.predict_proba(test_df[NAMES])[:, list(clf.classes_).index(1)]

baseline = LogisticRegression(max_iter=500)
baseline.fit(X_train, y_train)
train_df["p_base"] = baseline.predict_proba(X_train)[:, list(baseline.classes_).index(1)]
test_df["p_base"] = baseline.predict_proba(test_df[NAMES])[:, list(baseline.classes_).index(1)]
print("fitted", len(train_df), "rows")


fitted 3100 rows


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [10]:
def is_skip(row, p, tau):
    if row.task_type == "math":
        return False
    if bool(row.hard_skip):
        return True
    return float(p) < tau


def skip_counts(df, p_col, tau):
    skipped = [is_skip(row, getattr(row, p_col), tau) for row in df.itertuples(index=False)]
    skipped = np.asarray(skipped)
    solvable = df["solvable"].to_numpy()
    false_skips = int((skipped & solvable).sum())
    caught = int((skipped & ~solvable).sum())
    return false_skips, caught, skipped


def choose_tau(df, p_col):
    scores = df[p_col].to_numpy()
    candidates = sorted(set([0.0, 1.0, *scores.tolist()]))
    best = None
    for tau in candidates:
        false_skips, caught, _ = skip_counts(df, p_col, tau)
        # Zero false skips first, then as many true skips as possible.
        key = (false_skips == 0, -false_skips, caught)
        if best is None or key > best[0]:
            best = (key, tau, false_skips, caught)
    return best[1], best[2], best[3]


def stricter_tau(df, p_col, tau):
    """Higher threshold that removes false skips on this frame. Not saved."""
    scores = sorted(set([tau, 1.0, *df[p_col].tolist()]))
    scores = [value for value in scores if value >= tau]
    found = None
    for candidate in scores:
        false_skips, caught, _ = skip_counts(df, p_col, candidate)
        if false_skips == 0 and (found is None or caught > found[1] or (caught == found[1] and candidate < found[0])):
            found = (candidate, caught)
    return found


tau, train_fp, train_caught = choose_tau(train_df, "p")
print(f"tau={tau:.6f} train false_skips={train_fp} caught_unsolvable={train_caught}")

import joblib

OUT = out_dir()
SOLVABILITY_PATH = OUT / "solvability.joblib"
TAU_PATH = OUT / "tau.json"
joblib.dump(
    {
        "model": clf,
        "feature_names": NAMES,
        "schema_version": features.FEATURE_SCHEMA_VERSION,
    },
    SOLVABILITY_PATH,
)
TAU_PATH.write_text(
    json.dumps({"tau": tau, "feature_schema_version": features.FEATURE_SCHEMA_VERSION}),
    encoding="utf-8",
)
print("wrote", SOLVABILITY_PATH)
print("wrote", TAU_PATH)


tau=0.499492 train false_skips=0 caught_unsolvable=526
wrote /kaggle/working/solvability.joblib
wrote /kaggle/working/tau.json


## Metrics

False skip = a solvable task was not run. The goal is 0. Structural false skips are printed separately from the probability threshold.


In [11]:
def report(title, df, p_col, tau):
    false_skips, caught, skipped = skip_counts(df, p_col, tau)
    unsolvable = ~df["solvable"].to_numpy()
    predicted_unsolvable = skipped
    precision = caught / int(predicted_unsolvable.sum()) if predicted_unsolvable.any() else float("nan")
    recall = caught / int(unsolvable.sum()) if unsolvable.any() else float("nan")
    structural_fp = int(((df["hard_skip"]) & (df["solvable"]) & (df["task_type"] != "math")).sum())
    math_skips = int(sum(
        is_skip(row, getattr(row, p_col), tau)
        for row in df.itertuples(index=False)
        if row.task_type == "math"
    ))
    print(f"\n== {title} tau={tau:.6f} ==")
    print(f"false_skips={false_skips}  (goal 0)")
    print(f"unsolvable_precision={precision:.4f}  unsolvable_recall={recall:.4f}")
    print(f"structural_false_skips={structural_fp}  math_skips={math_skips}")
    frame = df.copy()
    frame["skip"] = skipped
    frame["pred_unsolvable"] = frame["skip"]
    print("by task_type:")
    for task_type, part in frame.groupby("task_type"):
        part_fp = int((part["skip"] & part["solvable"]).sum())
        part_caught = int((part["skip"] & ~part["solvable"]).sum())
        print(f"  {task_type}: false_skips={part_fp} caught={part_caught} n={len(part)}")
    matrix = pd.crosstab(
        frame["solvable"].map({True: "solvable", False: "unsolvable"}),
        frame["pred_unsolvable"].map({True: "skip", False: "run"}),
    )
    print(matrix)


report("train-mix HGB", train_df, "p", tau)
report("test-100 HGB", test_df, "p", tau)
test_fp, _, _ = skip_counts(test_df, "p", tau)
if test_fp:
    alt = stricter_tau(test_df, "p", tau)
    print("WARNING: test-100 has false skips. tau.json was NOT changed.")
    if alt is None:
        print("No stricter tau removes them. Structural skips are the cause if structural_false_skips > 0.")
    else:
        print(f"Stricter tau that zeroes test-100 false skips: {alt[0]:.6f} (caught {alt[1]}). Not saved.")

base_tau, base_fp, base_caught = choose_tau(train_df, "p_base")
print(f"\nlogistic baseline own tau={base_tau:.6f} train false_skips={base_fp} caught={base_caught}")
report("train-mix logistic", train_df, "p_base", base_tau)
report("test-100 logistic", test_df, "p_base", base_tau)



== train-mix HGB tau=0.499492 ==
false_skips=0  (goal 0)
unsolvable_precision=1.0000  unsolvable_recall=0.7514
structural_false_skips=0  math_skips=0
by task_type:
  coding: false_skips=0 caught=150 n=750
  data_analysis: false_skips=0 caught=133 n=750
  math: false_skips=0 caught=0 n=757
  search: false_skips=0 caught=243 n=843
pred_unsolvable   run  skip
solvable                   
solvable         2400     0
unsolvable        174   526

== test-100 HGB tau=0.499492 ==
false_skips=0  (goal 0)
unsolvable_precision=1.0000  unsolvable_recall=0.7000
structural_false_skips=0  math_skips=0
by task_type:
  coding: false_skips=0 caught=5 n=25
  data_analysis: false_skips=0 caught=4 n=25
  math: false_skips=0 caught=0 n=25
  search: false_skips=0 caught=5 n=25
pred_unsolvable  run  skip
solvable                  
solvable          80     0
unsolvable         6    14

logistic baseline own tau=0.086455 train false_skips=0 caught=408

== train-mix logistic tau=0.086455 ==
false_skips=0  (goal 